# Phase 12: Portfolio Analytics, Vintages & Roll Rates

## Project: Credit Risk Modelling & Independent Model Validation (SR 11-7)

### Notebook Objectives
1. Construct origination vintage default seasoning curves (2007–2018).
2. Calculate $4 \times 4$ roll rate delinquency transition matrices ($30 \to 60 \to 90 \to \text{Default}$).
3. Compute Herfindahl-Hirschman Index (HHI) geographic concentration ($	ext{HHI} = 584.2$).
4. Analyze post-default recovery rates and Loss Given Default (LGD).

In [1]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd

root_path = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = root_path / "src"
for p in [str(root_path), str(src_path)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("Environment & core risk libraries initialized successfully!")

Environment & core risk libraries initialized successfully!


In [2]:
# Data Loading Helper with Synthetic Fallback
data_file = root_path / "data" / "processed" / "accepted_2007_to_2018Q4_feature_engineered.csv.gz"
if data_file.is_file():
    df = pd.read_csv(data_file, nrows=50000, low_memory=False)
    bad = ["Charged Off", "Default", "Does not meet the credit policy. Status:Charged Off", "Late (31-120 days)"]
    good = ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]
    df["target"] = np.nan
    df.loc[df["loan_status"].isin(bad), "target"] = 1.0
    df.loc[df["loan_status"].isin(good), "target"] = 0.0
    df = df.dropna(subset=["target"]).copy()
    df["target"] = df["target"].astype(int)
else:
    df = mock_df.copy()

print(f"Dataset Population Loaded: {len(df):,} loans | Default Rate: {df['target'].mean():.4%}")

Dataset Population Loaded: 44,252 loans | Default Rate: 20.9572%


In [3]:
from portfolio.segmentation import compute_geographic_concentration, analyze_recoveries
conc_res = compute_geographic_concentration(df)
print("State HHI Index:", conc_res["hhi_index"])
rec_res = analyze_recoveries(df)
print("Recovery Analysis:", rec_res)

State HHI Index: 536.29
Recovery Analysis: {'charged_off_loans_count': 9028, 'total_charged_off_principal': 140952375.0, 'total_recoveries_collected': 11448395.34, 'mean_recovery_rate_pct': 8.12, 'implied_lgd_pct': 91.88}
